<img src="assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# DBSCAN Clustering

---

### About
Understand and use DBSCAN clustering.

### Learning Objectives
- Describe the effect of epsilon and min_points on DBSCAN.
- Identify advantages and disadvantages of DBSCAN.
- Implement DBSCAN using `scikit-learn`.

### Notebook Guide
- Imports
- Load Data
- Fit a DBSCAN model
- Conclusions and Takeaways

# Imports

In [ ]:
%pip install -qqq numpy pandas scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn import metrics

# Load data
---

In [ ]:
names = [
    'area', 'perimeter', 'compactness',
    'kern_len', 'kern_width', 'asymmetry',
    'groove_width', 'variety'
]
seeds = pd.read_csv('data/seeds_dataset.txt', sep=r'\s+', names=names)
seeds.head()

In [ ]:
# Some fast EDA
sns.pairplot(seeds, hue='variety', palette='Dark2');

In [ ]:
# Let's use ONLY kernel width and groove width to do our clustering.
# (YOU) make me an X and standard scale it!
X = seeds[['kern_width', 'groove_width']]
Z = StandardScaler().fit_transform(X)

# Fit a DBSCAN model
---

In [ ]:
# Let's fit a DBSCAN with epsilon = 0.3 and n = 7
db = DBSCAN(eps=0.3, min_samples=7)
db.fit(Z)

In [ ]:
# Let's plot our clusters on the original data scale and color them in.
plt.scatter(X.kern_width, X.groove_width, c=db.labels_)

## Model Evaluation: Silhouette Score
---

Recall the formula for silhouette score:

### $s_i = \frac{b_i - a_i}{max\{a_i, b_i\}}$

Where:
- $a_i$ = Cohesion: Average distance of points within clusters
- $b_i$ = Separation: Average distance from point $x_i$ to all points in the next nearest cluster.

In the cell below, use the `silhouette_score` function from `scikit-learn` to evaluate our `DBSCAN` model.

In [ ]:
# What's the silhouette score of our DBSCAN(0.3, 7) cluster?

metrics.silhouette_score(Z, db.labels_)

In [ ]:
# Put everything in one cell here and rapidly iterate epsilon and cluster

db = DBSCAN(eps=0.1, min_samples=5)
db.fit(Z)
plt.scatter(X.kern_width, X.groove_width, c=db.labels_)
metrics.silhouette_score(Z, db.labels_)

In [ ]:
# Painful, but you *could* brute force the params

e_vec = np.linspace(0.01, 0.7, 100)
n_vec = np.arange(1, 15)
res = []

for e in e_vec:
    for n in n_vec:
        cl = DBSCAN(eps=e, min_samples=n)
        cl.fit(Z)
        lbls = pd.Series(cl.labels_)
        if lbls.nunique() <= 1:
            continue
            
        sil = metrics.silhouette_score(Z, cl.labels_)
        res.append((e, n, sil))

In [ ]:
df_res = pd.DataFrame(res, columns=['e', 'n', 'sil'])

In [ ]:
df_res.sort_values('sil', ascending=False)

In [ ]:
db = DBSCAN(eps=0.54, min_samples=14)
db.fit(Z)
plt.scatter(X.kern_width, X.groove_width, c=db.labels_)
metrics.silhouette_score(Z, db.labels_)

## DBSCAN conclusions
* DBSCAN is an interesting one in that it clusters based on the density of clusters...
* However, it comes at the cost of needing TWO (2!) tuning parameters.
* DBSCAN can also help us detect **outliers**.